# 📚 Video2QA — T5-base Question & Answer Generation

Fine-tunes **T5-base** to take any input text and return `Q: ... A: ...` pairs.

**Features:**
- Input text is automatically chunked if too long
- Generates both factual and conceptual questions
- Answers are direct quotes from the input text
- Number of questions scales with text length (short: 3-4, medium: 5-7, long: 8-11)
- Duplicate questions are filtered out within a single run
- Frequent checkpoints → safe to resume after Colab disconnects
- Training uses SQuAD + Natural Questions for diverse coverage

## 🔄 How to Resume After a Crash
Just re-run all cells top to bottom. The resume cell detects the latest checkpoint automatically.

## 📦 Step 1 — Install Dependencies

In [ ]:
# Install all required libraries.
# rouge_score is used to detect duplicate/near-duplicate generated questions.
!pip install transformers datasets accelerate sentencepiece rouge_score -q

  Preparing metadata (setup.py) ... done


## 💾 Step 2 — Mount Google Drive

In [ ]:
# All checkpoints and the final model are saved to Google Drive.
# This ensures they survive Colab session resets.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## ⚙️ Step 3 — Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════
# CONFIGURATION — edit these values to customise training
# ══════════════════════════════════════════════════════════════

# T5-base: 220M parameters, much better quality than t5-small.
# Fits on free Colab T4 with the batch sizes below.
MODEL_NAME = "t5-base"

# Google Drive folder for all checkpoints + final model.
CHECKPOINT_DIR = "/content/drive/MyDrive/video2qa-t5base"

# ── Dataset ──────────────────────────────────────────────────
# We mix SQuAD (Wikipedia Q&A) + Natural Questions (real Google searches).
# Both give factual + contextual Q&A pairs that transfer well to
# video transcripts and educational content.
SQUAD_EXAMPLES = 70000     # out of 87,599 available
NQ_EXAMPLES    = 20000     # out of ~100k available (short answers only)

# ── Checkpoint settings ───────────────────────────────────────
# Save a checkpoint every N optimizer steps.
# 200 is safe for free Colab (disconnects every ~1.5h).
SAVE_EVERY_STEPS  = 200
KEEP_CHECKPOINTS  = 3      # keep N most recent + always keep best

# ── Training hyperparameters ──────────────────────────────────
EPOCHS         = 3         # 3 epochs gives noticeably better results than 2
BATCH_SIZE     = 4         # safe for T5-base on T4 GPU; increase to 16 if GPU RAM allows
GRAD_ACCUM     = 4         # effective batch = BATCH_SIZE * GRAD_ACCUM = 16
LEARNING_RATE  = 5e-5
LOG_STEPS      = 100

# ── Inference defaults (used at the bottom of this notebook) ─
# Text length thresholds (in words) for deciding how many questions to generate.
SHORT_WORD_LIMIT  = 200    # below this → 3-4 questions
MEDIUM_WORD_LIMIT = 500    # below this → 5-7 questions, above → 8-11

# Chunk size when splitting long texts (in words).
# Each chunk is processed independently and questions are merged + deduped.
CHUNK_SIZE_WORDS = 300

# Similarity threshold for duplicate filtering (0–1, higher = stricter).
# Questions with ROUGE-L score above this are considered duplicates.
DEDUP_THRESHOLD = 0.7

print("Config loaded.")
print(f"  Model          : {MODEL_NAME}")
print(f"  Checkpoint dir : {CHECKPOINT_DIR}")
print(f"  Dataset        : SQuAD ({SQUAD_EXAMPLES:,}) + NQ ({NQ_EXAMPLES:,})")
print(f"  Epochs         : {EPOCHS}  |  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Save every     : {SAVE_EVERY_STEPS} steps")

Config loaded.
  Model          : t5-base
  Checkpoint dir : /content/drive/MyDrive/video2qa-t5base
  Dataset        : SQuAD (70,000) + NQ (20,000)
  Epochs         : 3  |  Effective batch: 16
  Save every     : 200 steps


## 🔧 Step 4 — Imports

In [ ]:
import os
import re
import glob
import torch
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset, concatenate_datasets
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   : {DEVICE}")

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4
VRAM     : 15.6 GB
Device   : cuda


## 🔍 Step 5 — Checkpoint Utilities

In [ ]:
# ─────────────────────────────────────────────────────────────
# Checkpoint helpers
# ─────────────────────────────────────────────────────────────

def find_latest_checkpoint(checkpoint_dir: str):
    """
    Return the path to the checkpoint with the highest step number,
    or None if no checkpoints exist yet.
    """
    pattern = os.path.join(checkpoint_dir, "checkpoint-*")
    checkpoints = glob.glob(pattern)
    if not checkpoints:
        return None
    return max(checkpoints, key=lambda p: int(os.path.basename(p).split("-")[-1]))


def list_all_checkpoints(checkpoint_dir: str):
    """Print a table of all saved checkpoints with step number and size."""
    pattern = os.path.join(checkpoint_dir, "checkpoint-*")
    checkpoints = sorted(
        glob.glob(pattern),
        key=lambda p: int(os.path.basename(p).split("-")[-1])
    )
    if not checkpoints:
        print("  (no checkpoints found)")
        return
    print(f"  {'Step':>8}   {'Size':>8}   Path")
    print(f"  {'----':>8}   {'----':>8}   ----")
    for ckpt in checkpoints:
        step = os.path.basename(ckpt).split("-")[-1]
        total = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(ckpt) for f in files
        )
        print(f"  {step:>8}   {total/1e6:>6.0f}MB   {ckpt}")


# ── Inspect current state ─────────────────────────────────────
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
latest_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)

print(f"Checkpoint directory: {CHECKPOINT_DIR}")
print()
list_all_checkpoints(CHECKPOINT_DIR)
print()

if latest_ckpt:
    step = int(os.path.basename(latest_ckpt).split("-")[-1])
    print(f"✅ Found checkpoint at step {step} → will RESUME")
else:
    print("🆕 No checkpoint found → will train from SCRATCH")

Checkpoint directory: /content/drive/MyDrive/video2qa-t5base

      Step       Size   Path
      ----       ----   ----
      5000      896MB   /content/drive/MyDrive/video2qa-t5base/checkpoint-5000
      6200      896MB   /content/drive/MyDrive/video2qa-t5base/checkpoint-6200
      6400      896MB   /content/drive/MyDrive/video2qa-t5base/checkpoint-6400
      6600      892MB   /content/drive/MyDrive/video2qa-t5base/checkpoint-6600

✅ Found checkpoint at step 6600 → will RESUME


## 🤖 Step 6 — Load Model & Tokenizer

In [ ]:
# Load T5-base tokenizer and model from HuggingFace Hub.
#
# We always load the base pretrained weights here.
# When trainer.train(resume_from_checkpoint=...) is called later,
# the Trainer replaces these weights with the checkpoint's saved weights
# and restores optimizer state, so training continues seamlessly.

print(f"Loading tokenizer: {MODEL_NAME} ...")
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model: {MODEL_NAME} ...")
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)

num_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"\n✅ Model ready: {MODEL_NAME} ({num_params:.0f}M parameters)")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU memory used: {used:.1f} / {total:.1f} GB")

Loading tokenizer: t5-base ...


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading model: t5-base ...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]


✅ Model ready: t5-base (223M parameters)
   GPU memory used: 0.9 / 15.6 GB


## 📚 Step 7 — Load & Mix Datasets

In [ ]:
# ─────────────────────────────────────────────────────────────
# Dataset strategy
#
# We combine two complementary datasets:
#
# 1. SQuAD v1.1 — Wikipedia passages with factual + span-based answers.
#    Good for: grounded answers, factual questions.
#
# 2. Natural Questions (NQ) — real questions people typed into Google,
#    answered from Wikipedia. More diverse and conversational.
#    Good for: natural-sounding questions, broader coverage.
#
# Both datasets provide: context passage, question, answer span.
# We normalise them to the same format before tokenising.
# ─────────────────────────────────────────────────────────────

print("Loading SQuAD ...")
squad_raw = load_dataset("squad")["train"].select(range(SQUAD_EXAMPLES))

print("Loading Natural Questions ...")
# NQ has a 'validation' split that's cleaner for our purpose.
# We use the 'nq_open' version which has clean short answers.
nq_raw = load_dataset("nq_open", split="train").select(range(NQ_EXAMPLES))

print(f"\nSQuAD examples  : {len(squad_raw):,}")
print(f"NQ examples     : {len(nq_raw):,}")
print(f"Total           : {len(squad_raw) + len(nq_raw):,}")

Loading SQuAD ...


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

...

nq_open/train-00000-of-00001.parquet:   0%|          | 0.00/4.46M [00:00<?, ?B/s]

nq_open/validation-00000-of-00001.parque(…):   0%|          | 0.00/214k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87925 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3610 [00:00<?, ? examples/s]


SQuAD examples  : 70,000
NQ examples     : 20,000
Total           : 90,000


In [ ]:
# ─────────────────────────────────────────────────────────────
# Preprocessing
#
# Input format  : "generate question and answer: <context>"
# Target format : "Q: <question> A: <answer>"
#
# This teaches T5 to output both question and answer together,
# matching exactly the output format we want at inference time.
#
# Why this input format?
#   The prefix "generate question and answer:" is a T5 task prefix.
#   T5 was pretrained to follow such prefixes, so this guides the model
#   to understand what kind of output is expected.
# ─────────────────────────────────────────────────────────────

MAX_INPUT_LENGTH  = 512   # max tokens for context input
MAX_TARGET_LENGTH = 128   # max tokens for Q+A output


def preprocess_squad(example):
    """Preprocess a single SQuAD example."""
    if not example["answers"]["text"]:
        return None

    context  = example["context"]
    question = example["question"]
    answer   = example["answers"]["text"][0]  # first answer span

    # Verify the answer actually appears in the context (direct quote check)
    if answer.lower() not in context.lower():
        return None

    input_text  = f"generate question and answer: {context}"
    target_text = f"Q: {question} A: {answer}"

    return _tokenize(input_text, target_text)


def preprocess_nq(example):
    """
    Preprocess a Natural Questions example.
    NQ 'nq_open' has: question + list of short answers (no passage).
    We use the question as-is and the first short answer.
    Since NQ-open has no context passage, we use the answer as a mini-context.
    This teaches the model to handle short factual snippets too.
    """
    if not example.get("answer"):
        return None

    question = example["question"]
    answer   = example["answer"][0] if isinstance(example["answer"], list) else example["answer"]

    # Use a minimal context: just the answer itself as a snippet
    context = answer

    input_text  = f"generate question and answer: {context}"
    target_text = f"Q: {question} A: {answer}"

    return _tokenize(input_text, target_text)


def _tokenize(input_text: str, target_text: str):
    """Shared tokenisation logic for all datasets."""
    model_input = tokenizer(
        input_text,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        target_text,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    # Replace pad token IDs with -100 so the loss ignores padding positions
    model_input["labels"] = [
        t if t != tokenizer.pad_token_id else -100
        for t in labels["input_ids"]
    ]
    return model_input


# ── Apply preprocessing ───────────────────────────────────────
print("Tokenising SQuAD ...")
tok_squad = squad_raw.map(
    preprocess_squad,
    remove_columns=squad_raw.column_names,
    desc="SQuAD",
).filter(lambda x: x is not None)

print("Tokenising Natural Questions ...")
tok_nq = nq_raw.map(
    preprocess_nq,
    remove_columns=nq_raw.column_names,
    desc="NQ",
).filter(lambda x: x is not None)

# Merge both datasets
train_dataset = concatenate_datasets([tok_squad, tok_nq])
train_dataset = train_dataset.shuffle(seed=42)  # mix SQuAD and NQ together

print(f"\n✅ Final training set: {len(train_dataset):,} examples")
print(f"   SQuAD : {len(tok_squad):,}")
print(f"   NQ    : {len(tok_nq):,}")

Tokenising SQuAD ...


SQuAD:   0%|          | 0/70000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/70000 [00:00<?, ? examples/s]

Tokenising Natural Questions ...


NQ:   0%|          | 0/20000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]


✅ Final training set: 90,000 examples
   SQuAD : 70,000
   NQ    : 20,000


## 🏋️ Step 8 — Configure Trainer

In [ ]:
# ─────────────────────────────────────────────────────────────
# Validation set
# We use SQuAD validation split (1,500 examples) to track
# eval loss at each checkpoint and keep the best model.
# ─────────────────────────────────────────────────────────────

print("Preparing validation set ...")
val_raw = load_dataset("squad")["validation"].select(range(1500))
val_dataset = val_raw.map(
    preprocess_squad,
    remove_columns=val_raw.column_names,
    desc="Val",
).filter(lambda x: x is not None)
print(f"Validation examples: {len(val_dataset):,}")

# ─────────────────────────────────────────────────────────────
# Data collator — dynamic padding per batch
# ─────────────────────────────────────────────────────────────

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    pad_to_multiple_of=8,  # aligns tensor sizes for faster CUDA matmuls
)

# ─────────────────────────────────────────────────────────────
# TrainingArguments
# ─────────────────────────────────────────────────────────────

training_args = TrainingArguments(
    # Output
    output_dir=CHECKPOINT_DIR,
    lr_scheduler_type = "cosine",
    warmup_steps = 200,

    # Checkpoint frequency
    save_strategy="steps",
    save_steps=SAVE_EVERY_STEPS,
    save_total_limit=KEEP_CHECKPOINTS,  # + best is always kept

    # Evaluation (must match save strategy to enable best-model tracking)
    eval_strategy="steps",
    eval_steps=SAVE_EVERY_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Training hyperparameters
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,

    # Speed optimisations
    fp16=torch.cuda.is_available(),       # half-precision on GPU
    dataloader_num_workers=2,             # parallel data loading
    dataloader_pin_memory=True,           # faster CPU→GPU transfer
    optim="adafactor",                    # T5's native optimizer, memory efficient

    # Logging
    logging_strategy="steps",
    logging_steps=LOG_STEPS,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
)

print("\n✅ Trainer ready.")
print(f"   Effective batch size : {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Save every           : {SAVE_EVERY_STEPS} steps")
print(f"   Keep checkpoints     : {KEEP_CHECKPOINTS} + best")

Preparing validation set ...
Validation examples: 1,500

✅ Trainer ready.
   Effective batch size : 16
   Save every           : 200 steps
   Keep checkpoints     : 3 + best


## 🚀 Step 9 — Train (auto-resume)

**First run** → trains from scratch.  
**After a crash** → re-run all cells, this cell resumes automatically.

In [ ]:
resume_from = find_latest_checkpoint(CHECKPOINT_DIR)

if resume_from:
    step = int(os.path.basename(resume_from).split("-")[-1])
    print(f"🔁 Resuming from step {step}")
    print(f"   {resume_from}")
else:
    print("🆕 Starting training from scratch")

print(f"   Total epochs : {EPOCHS}")
print(f"   Save every   : {SAVE_EVERY_STEPS} steps")
print()

train_result = trainer.train(resume_from_checkpoint=resume_from)

print("\n✅ Training complete!")
print(f"   Runtime : {train_result.metrics.get('train_runtime', 0) / 3600:.2f}h")

🔁 Resuming from step 6400
   /content/drive/MyDrive/video2qa-t5base/checkpoint-6400
   Total epochs : 3
   Save every   : 200 steps



There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Step,Training Loss,Validation Loss
6600,7.593742,1.465821
6800,7.280392,1.437877
7000,7.225031,1.417902
7200,7.025582,1.400331
7400,6.989266,1.386365
7600,6.945574,1.376861
7800,6.947980,1.369885
8000,6.864450,1.364149
8200,6.949846,1.359319
8400,6.782130,1.356208


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



✅ Training complete!
   Runtime : 2.61h


## 💾 Step 10 — Save Final Model

In [ ]:
# Saves the best model (lowest eval loss) and tokenizer to Drive.
# This is the directory you'll point your other project at.

FINAL_DIR = os.path.join(CHECKPOINT_DIR, "final")
os.makedirs(FINAL_DIR, exist_ok=True)

trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print(f"✅ Final model saved to: {FINAL_DIR}")
print(f"   Load with: T5ForConditionalGeneration.from_pretrained('{FINAL_DIR}')")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Final model saved to: /content/drive/MyDrive/video2qa-t5base/final
   Load with: T5ForConditionalGeneration.from_pretrained('/content/drive/MyDrive/video2qa-t5base/final')


## 🧪 Step 11 — Inference Engine

This is the complete inference pipeline your other project will use.
Given any text, it returns `Q: ... A: ...` pairs.

In [ ]:
# ─────────────────────────────────────────────────────────────
# (Optional) Reload saved model in a fresh session
# Skip this cell if you just finished training (model is already in memory)
# ─────────────────────────────────────────────────────────────

RELOAD_PATH = os.path.join(CHECKPOINT_DIR, "final")

if os.path.exists(RELOAD_PATH):
    print(f"Loading model from: {RELOAD_PATH}")
    tokenizer = T5Tokenizer.from_pretrained(RELOAD_PATH)
    model     = T5ForConditionalGeneration.from_pretrained(RELOAD_PATH).to(DEVICE)
    print("✅ Model loaded.")
else:
    print("Using in-memory model from training.")

Loading model from: /content/drive/MyDrive/video2qa-t5base/final


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

✅ Model loaded.


In [ ]:
import re
import glob
import torch
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset, concatenate_datasets
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)


# ══════════════════════════════════════════════════════════════
# INFERENCE ENGINE
# This is the code you'll copy into your other project.
# ══════════════════════════════════════════════════════════════

from rouge_score import rouge_scorer as rouge_lib

_rouge = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)


# ── Helpers ───────────────────────────────────────────────────

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE_WORDS) -> list:
    """
    Split text into overlapping word-based chunks.

    Uses a 20% overlap between chunks so questions at chunk boundaries
    are still grounded in enough context.

    Args:
        text       : Raw input text.
        chunk_size : Target size of each chunk in words.

    Returns:
        List of text chunks (strings).
    """
    words = text.split()
    if len(words) <= chunk_size:
        return [text]  # short enough, no splitting needed

    overlap = max(1, chunk_size // 5)  # 20% overlap
    chunks  = []
    start   = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap  # step back by overlap for next chunk

    return chunks


def count_questions(text: str) -> int:
    """
    Decide how many Q&A pairs to generate based on total text length.

    Thresholds (configurable in Step 3):
        Short  (< SHORT_WORD_LIMIT words)  → 3–4 questions
        Medium (< MEDIUM_WORD_LIMIT words) → 5–7 questions
        Long   (>= MEDIUM_WORD_LIMIT)      → 8–11 questions
    """
    word_count = len(text.split())
    if word_count < SHORT_WORD_LIMIT:
        return 3   # short: aim for 3 (will generate up to 4 before dedup)
    elif word_count < MEDIUM_WORD_LIMIT:
        return 5   # medium: aim for 5 (up to 7)
    else:
        return 8   # long: aim for 8 (up to 11)


def is_duplicate(new_q: str, existing_qs: list, threshold: float = DEDUP_THRESHOLD) -> bool:
    """
    Return True if new_q is too similar to any question already in existing_qs.

    Uses ROUGE-L (longest common subsequence) as the similarity metric.
    ROUGE-L > threshold means the questions share too many words in the same order.
    """
    for existing in existing_qs:
        score = _rouge.score(new_q.lower(), existing.lower())["rougeL"].fmeasure
        if score >= threshold:
            return True
    return False


def parse_qa(raw_output: str):
    """
    Parse the model's raw output string into a (question, answer) tuple.

    Expected format: "Q: <question text> A: <answer text>"

    Returns:
        (question_str, answer_str) or None if parsing fails.
    """
    # Match "Q: ... A: ..." pattern, case-insensitive
    match = re.search(r"Q:\s*(.+?)\s+A:\s*(.+)", raw_output, re.IGNORECASE | re.DOTALL)
    if not match:
        return None
    question = match.group(1).strip()
    answer   = match.group(2).strip()
    return question, answer


def generate_qa_from_chunk(
    chunk: str,
    num_questions: int,
    num_beams: int = 5,
) -> list:
    """
    Generate `num_questions` Q&A pairs from a single text chunk.

    Uses diverse beam search so each returned sequence explores
    a different part of the answer space.

    Args:
        chunk         : A piece of input text (≤ CHUNK_SIZE_WORDS words).
        num_questions : How many Q&A pairs to request.
        num_beams     : Beam search width per group.

    Returns:
        List of (question, answer) tuples parsed from model output.
    """
    input_text = f"generate question and answer: {chunk}"

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(DEVICE)

    # Generate more candidates than needed so we have room to dedup
    n_candidates = num_questions + 3

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=n_candidates,
            num_return_sequences=n_candidates,
            num_beam_groups=n_candidates,      # diverse beam search
            diversity_penalty=0.8,             # push beams apart for variety
            early_stopping=True,
            trust_remote_code=True # Added this line
        )

    results = []
    for output in outputs:
        raw = tokenizer.decode(output, skip_special_tokens=True)
        parsed = parse_qa(raw)
        if parsed:
            results.append(parsed)

    return results


# ── Main public function ──────────────────────────────────────

def generate_qa(text: str, verbose: bool = False) -> list:
    """
    Main entry point. Given any input text, return a list of Q&A pairs.

    This is the function to import into your other project.

    Args:
        text    : Any English text — transcript, article, educational content.
        verbose : If True, print progress info during generation.

    Returns:
        List of dicts: [{"question": "...", "answer": "..."}, ...]

    Example:
        pairs = generate_qa("The Eiffel Tower was built in 1889 by Gustave Eiffel...")
        for pair in pairs:
            print(f"Q: {pair['question']}")
            print(f"A: {pair['answer']}")
    """
    text = text.strip()
    if not text:
        return []

    # 1. Decide total number of questions based on full text length
    total_needed = count_questions(text)
    if verbose:
        word_count = len(text.split())
        print(f"Text length : {word_count} words → targeting {total_needed} questions")

    # 2. Split into chunks if the text is long
    chunks = chunk_text(text)
    if verbose:
        print(f"Chunks      : {len(chunks)}")

    # 3. Distribute questions evenly across chunks
    questions_per_chunk = max(1, -(-total_needed // len(chunks)))  # ceiling division

    # 4. Generate candidates from each chunk
    all_candidates = []
    for i, chunk in enumerate(chunks):
        if verbose:
            print(f"Generating from chunk {i+1}/{len(chunks)} ...")
        candidates = generate_qa_from_chunk(chunk, questions_per_chunk)
        all_candidates.extend(candidates)

    # 5. Deduplicate: keep only questions that are distinct enough
    seen_questions = []
    final_pairs    = []

    for question, answer in all_candidates:
        if not question or not answer:
            continue
        if is_duplicate(question, seen_questions):
            continue
        seen_questions.append(question)
        final_pairs.append({"question": question, "answer": answer})

        # Stop once we have enough
        if len(final_pairs) >= total_needed:
            break

    if verbose:
        print(f"Generated   : {len(final_pairs)} unique Q&A pairs")

    return final_pairs


print("✅ Inference engine ready. Use generate_qa(text) to generate Q&A pairs.")

✅ Inference engine ready. Use generate_qa(text) to generate Q&A pairs.


## ✅ Step 12 — Test the Model

In [ ]:
# ── Test 1: Short text ────────────────────────────────────────

short_text = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
It was designed by Gustave Eiffel and built between 1887 and 1889 as the entrance arch
for the 1889 World's Fair. Standing at 330 metres tall, it was the world's tallest
man-made structure for 41 years.
"""

print("=" * 60)
print("SHORT TEXT TEST")
print("=" * 60)
pairs = generate_qa(short_text, verbose=True)
print()
for i, pair in enumerate(pairs, 1):
    print(f"{i}. Q: {pair['question']}")
    print(f"   A: {pair['answer']}")
    print()

SHORT TEXT TEST
Text length : 52 words → targeting 3 questions
Chunks      : 1
Generating from chunk 1/1 ...


generate.py: 0.00B [00:00, ?B/s]

beam_search.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/transformers-community/group-beam-search:
- custom_generate/beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/transformers-community/group-beam-search:
- custom_generate/generate.py
- beam_search.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'early_stopping', 'num_beams', 'num_beam_groups', 'num_return_sequences', 'diversity_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=128) and

Generated   : 3 unique Q&A pairs

1. Q: What was the height of the Eiffel Tower?
   A: 330 metres

2. Q: Who designed the Eiffel Tower?
   A: Gustave Eiffel

3. Q: How tall is the Eiffel Tower?
   A: 330 metres



In [ ]:
# ── Test 2: Long text (triggers chunking) ─────────────────────

long_text = """
Machine learning is a branch of artificial intelligence that focuses on building systems
that learn from data. Instead of being explicitly programmed, these systems improve their
performance through experience. The field was formally founded in 1956 at the Dartmouth
Conference, where researchers first coined the term artificial intelligence.

There are three main types of machine learning. Supervised learning uses labelled training
data to learn a mapping from inputs to outputs. Common examples include email spam detection
and image classification. Unsupervised learning finds hidden patterns in data without labels,
and is used in customer segmentation and anomaly detection. Reinforcement learning trains
agents to make decisions by rewarding correct actions, and has achieved superhuman
performance in games like Chess and Go.

Deep learning, a subfield of machine learning, uses neural networks with many layers to
learn complex representations. It has transformed computer vision, natural language
processing, and speech recognition. The rise of deep learning was enabled by three factors:
large datasets, powerful GPUs, and algorithmic innovations like the backpropagation algorithm.
Today, deep learning models like GPT and BERT underpin most modern AI applications.
"""

print("=" * 60)
print("LONG TEXT TEST (with chunking)")
print("=" * 60)
pairs = generate_qa(long_text, verbose=True)
print()
for i, pair in enumerate(pairs, 1):
    print(f"{i}. Q: {pair['question']}")
    print(f"   A: {pair['answer']}")
    print()

LONG TEXT TEST (with chunking)
Text length : 181 words → targeting 3 questions
Chunks      : 1
Generating from chunk 1/1 ...


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated   : 3 unique Q&A pairs

1. Q: What is a subfield of machine learning that uses neural networks with many layers to learn complex representations?
   A: Deep learning

2. Q: What is the term used to describe machine learning?
   A: Unsupervised learning

3. Q: What does machine learning focus on?
   A: building systems that learn from data



In [ ]:
# ── Test 3: Try your own text ─────────────────────────────────
# Paste any video transcript, article, or educational text here.

your_text = """
Paste your own text here.
"""

pairs = generate_qa(your_text, verbose=True)
print()
for i, pair in enumerate(pairs, 1):
    print(f"{i}. Q: {pair['question']}")
    print(f"   A: {pair['answer']}")
    print()

Text length : 5 words → targeting 3 questions
Chunks      : 1
Generating from chunk 1/1 ...


Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated   : 3 unique Q&A pairs

1. Q: what is the name of the page where you can paste your own text
   A: Paste your own text here

2. Q: where do you paste your own text
   A: Paste your own text here

3. Q: how do you add a link to your own text
   A: Paste your own text here



## 📦 Step 13 — Export Inference Code

Copy this snippet into your other project.
Point `MODEL_PATH` at your saved model directory on Drive.

In [ ]:
# ══════════════════════════════════════════════════════════════
# COPY THIS INTO YOUR OTHER PROJECT
# ══════════════════════════════════════════════════════════════

snippet = '''
import re
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from rouge_score import rouge_scorer as rouge_lib

# ── Config ────────────────────────────────────────────────────
MODEL_PATH        = "/content/drive/MyDrive/video2qa-t5base/final"
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
MAX_INPUT_LENGTH  = 512
MAX_TARGET_LENGTH = 128
CHUNK_SIZE_WORDS  = 300
DEDUP_THRESHOLD   = 0.7
SHORT_WORD_LIMIT  = 200
MEDIUM_WORD_LIMIT = 500

# ── Load model (do this once at startup) ─────────────────────
tokenizer = T5Tokenizer.from_pretrained(MODEL_PATH)
model     = T5ForConditionalGeneration.from_pretrained(MODEL_PATH).to(DEVICE)
model.eval()
_rouge = rouge_lib.RougeScorer(["rougeL"], use_stemmer=True)

# ── Helpers (same as training notebook) ──────────────────────
def chunk_text(text, chunk_size=CHUNK_SIZE_WORDS):
    words = text.split()
    if len(words) <= chunk_size:
        return [text]
    overlap, chunks, start = max(1, chunk_size // 5), [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words): break
        start = end - overlap
    return chunks

def count_questions(text):
    n = len(text.split())
    return 3 if n < SHORT_WORD_LIMIT else 5 if n < MEDIUM_WORD_LIMIT else 8

def is_duplicate(q, existing, threshold=DEDUP_THRESHOLD):
    return any(_rouge.score(q.lower(), e.lower())["rougeL"].fmeasure >= threshold for e in existing)

def parse_qa(raw):
    m = re.search(r"Q:\\s*(.+?)\\s+A:\\s*(.+)", raw, re.IGNORECASE | re.DOTALL)
    return (m.group(1).strip(), m.group(2).strip()) if m else None

def generate_qa(text):
    """Main function. Input: text string. Output: list of {question, answer} dicts."""
    text = text.strip()
    if not text: return []
    total_needed = count_questions(text)
    chunks = chunk_text(text)
    qpc = max(1, -(-total_needed // len(chunks)))
    all_candidates = []
    for chunk in chunks:
        inputs = tokenizer(f"generate question and answer: {chunk}",
                           return_tensors="pt", max_length=MAX_INPUT_LENGTH,
                           truncation=True).to(DEVICE)
        n = qpc + 3
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=MAX_TARGET_LENGTH,
                                     num_beams=n, num_return_sequences=n,
                                     num_beam_groups=n, diversity_penalty=0.8,
                                     early_stopping=True)
        for o in outputs:
            p = parse_qa(tokenizer.decode(o, skip_special_tokens=True))
            if p: all_candidates.append(p)
    seen, final = [], []
    for q, a in all_candidates:
        if not q or not a or is_duplicate(q, seen): continue
        seen.append(q)
        final.append({"question": q, "answer": a})
        if len(final) >= total_needed: break
    return final
'''

print(snippet)
print("─" * 60)
print("Copy everything between the lines into your project.")
print("Then call: pairs = generate_qa(your_text)")

## 🗂️ Step 14 — Checkpoint Management

In [ ]:
import shutil

# Inspect all checkpoints
print(f"Checkpoints in: {CHECKPOINT_DIR}")
print()
list_all_checkpoints(CHECKPOINT_DIR)

latest = find_latest_checkpoint(CHECKPOINT_DIR)
if latest:
    print(f"\n→ Next run will resume from: {os.path.basename(latest)}")

In [ ]:
# (Optional) Delete old checkpoints to free Drive space.
# Set DRY_RUN = False to actually delete.

KEEP_LAST_N = 2
DRY_RUN     = True

all_ckpts = sorted(
    glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")),
    key=lambda p: int(os.path.basename(p).split("-")[-1])
)
to_delete = all_ckpts[:-KEEP_LAST_N] if len(all_ckpts) > KEEP_LAST_N else []

print(f"Would delete: {[os.path.basename(p) for p in to_delete]}")
print(f"Would keep  : {[os.path.basename(p) for p in all_ckpts[-KEEP_LAST_N:]]}")

if not DRY_RUN:
    for ckpt in to_delete:
        shutil.rmtree(ckpt)
        print(f"Deleted: {ckpt}")
else:
    print("\n[DRY RUN] Set DRY_RUN=False to actually delete.")